# Sampler Smoke Test (`emcee` / `zeus` / `dynesty`)

This notebook runs light-weight fit smoke tests for two modes:
- bolometric (`fit_bol`, model=`nickel`)
- multiband (`fit_multiband`, model=`nickel`)

It is intended only for API/backend sanity checks, not scientific inference quality.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parent if (Path.cwd() / 'data').exists() else Path.cwd().resolve()
if (ROOT / 'transfit').exists() and str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import transfit as tf

EXAMPLES_DIR = ROOT / 'examples'
DATA_DIR = EXAMPLES_DIR / 'data'
print('ROOT      =', ROOT)
print('DATA_DIR  =', DATA_DIR)


ROOT      = /Users/zyh/Desktop/Transfit_v0.4
DATA_DIR  = /Users/zyh/Desktop/Transfit_v0.4/examples/data


In [2]:
def load_bolometric_data() -> tf.BolometricData:
    p = DATA_DIR / 'sn1993j_lbol.txt'
    arr = np.loadtxt(p, comments='#')
    t_days = np.asarray(arr[:, 0], float)
    lbol = np.asarray(arr[:, 1], float)
    lbol_err = np.asarray(arr[:, 2], float)

    # Keep early-time points to make the smoke test fast.
    keep = (t_days >= 3.0) & (t_days <= 45.0)
    return tf.BolometricData(
        t_days=t_days[keep],
        y=lbol[keep],
        yerr=lbol_err[keep],
    )


def load_multiband_data() -> tf.MultiBandData:
    p = DATA_DIR / 'sn2007gr.csv'
    df = pd.read_csv(p)

    t_col = np.asarray(df['Phase'], float)
    bands, t_days, mags, errs = [], [], [], []

    mag_cols = {
        'B': ('Bmag', 'e_Bmag'),
        'V': ('Vmag', 'e_Vmag'),
        'R': ('Rmag', 'e_Rmag'),
        'I': ('Imag', 'e_Imag'),
    }

    for band, (mcol, ecol) in mag_cols.items():
        y = pd.to_numeric(df[mcol], errors='coerce').to_numpy(float)
        e = pd.to_numeric(df[ecol], errors='coerce').to_numpy(float)
        m = np.isfinite(t_col) & np.isfinite(y) & np.isfinite(e) & (e > 0)

        # Limit to early time for a fast smoke test.
        m &= (t_col >= -7.0) & (t_col <= 45.0)

        if not np.any(m):
            continue

        t_days.extend(t_col[m].tolist())
        mags.extend(y[m].tolist())
        errs.extend(e[m].tolist())
        bands.extend([band] * int(np.sum(m)))

    return tf.MultiBandData(
        t_days=np.asarray(t_days, float),
        band=np.asarray(bands, dtype=object),
        y=np.asarray(mags, float),
        yerr=np.asarray(errs, float),
    )


In [3]:
sampler_cfg = {
    'emcee': {
        'nwalkers': 12,
        'nsteps': 20,
        'burnin': 5,
        'thin': 1,
        'seed': 42,
        'progress': False,
        'robust_init': False,
    },
    'zeus': {
        'nwalkers': 12,
        'nsteps': 20,
        'burnin': 5,
        'thin': 1,
        'seed': 42,
        'progress': False,
        'robust_init': False,
    },
    'dynesty': {
        'nlive': 40,
        'maxiter': 120,
        'dlogz': 5.0,
        'nsamples': 200,
        'seed': 42,
        'progress': False,
        'sample': 'rwalk',
        'bound': 'multi',
    },
}


def run_one_bol_sampler(sampler: str, cfg: dict) -> str:
    data = load_bolometric_data()
    ctx = tf.Context(distance=tf.Distance(z=0.001))

    priors = {
        'M_ej': (0.5, 5.0),
        'v_ej': (0.2, 2.0),
        'M_Ni': (0.02, 0.8),
    }
    fixed = {
        'x_Ni': 0.5,
        'kappa': 0.12,
        'kappa_gamma': 0.03,
    }
    model_kwargs = {
        'Nx': 24,
        'Ny': 180,
        't_max_days': 60.0,
    }

    try:
        res = tf.fit_bol(
            data=data,
            model='nickel',
            ctx=ctx,
            priors=priors,
            fixed=fixed,
            sampler=sampler,
            sampler_kwargs=cfg,
            model_kwargs=model_kwargs,
            include_t_shift=False,
        )
    except ImportError as exc:
        print(f'[SKIP][bol] sampler={sampler}: {exc}')
        return 'skip'
    except Exception as exc:
        print(f'[FAIL][bol] sampler={sampler}: {exc}')
        return 'fail'

    ns, ndim = res.samples.shape
    best_logp = float(np.nanmax(res.log_prob))
    print(f'[OK][bol] sampler={sampler:7s} samples={ns} ndim={ndim} best_logp={best_logp:.3e}')
    return 'ok'


print('=== Bolometric smoke test (model=nickel) ===')
counts_bol = {'ok': 0, 'skip': 0, 'fail': 0}
for name in ('emcee', 'zeus', 'dynesty'):
    st = run_one_bol_sampler(name, sampler_cfg[name])
    counts_bol[st] += 1
print('Bolometric summary:', counts_bol)


=== Bolometric smoke test (model=nickel) ===


Initializing and JIT-compiling the model...
Model is ready for fast execution.
[OK][bol] sampler=emcee   samples=240 ndim=3 best_logp=-6.929e+04


Initialising ensemble of 12 walkers...


[OK][bol] sampler=zeus    samples=180 ndim=3 best_logp=-6.915e+04
[OK][bol] sampler=dynesty samples=200 ndim=3 best_logp=-7.252e+04
Bolometric summary: {'ok': 3, 'skip': 0, 'fail': 0}


/opt/anaconda3/lib/python3.12/site-packages/dynesty/dynesty.py:161: UserWarning: Specifying slice option while using rwalk sampler or  walks option with a slice sampler does not make sense
  warnings.warn('Specifying slice option while using rwalk sampler or '
/opt/anaconda3/lib/python3.12/site-packages/dynesty/sampler.py:1082: UserWarning: The sampling was stopped short due to maxiter/maxcall limit the delta(log(z)) criterion is not achieved; posterior may be poorly sampled
  warnings.warn('The sampling was stopped short due to'


In [4]:
def run_one_multiband_sampler(sampler: str, cfg: dict) -> str:
    data = load_multiband_data()

    filters = {
        'B': 6.8e14,
        'V': 5.5e14,
        'R': 4.7e14,
        'I': 3.9e14,
    }
    ctx = tf.Context(
        distance=tf.Distance(z=0.001728),
        filters=filters,
        y_kind='mag',
    )

    priors = {
        'M_ej': (0.5, 5.0),
        'v_ej': (0.2, 2.0),
        'M_Ni': (0.02, 0.8),
    }
    fixed = {
        'x_Ni': 0.5,
        'kappa': 0.12,
        'kappa_gamma': 0.03,
        'T_floor': 4500.0,
    }
    model_kwargs = {
        'Nx': 20,
        'Ny': 160,
        't_max_days': 60.0,
    }

    try:
        res = tf.fit_multiband(
            data=data,
            model='nickel',
            ctx=ctx,
            priors=priors,
            fixed=fixed,
            sampler=sampler,
            sampler_kwargs=cfg,
            model_kwargs=model_kwargs,
            include_t_shift=False,
        )
    except ImportError as exc:
        print(f'[SKIP][mb ] sampler={sampler}: {exc}')
        return 'skip'
    except Exception as exc:
        print(f'[FAIL][mb ] sampler={sampler}: {exc}')
        return 'fail'

    ns, ndim = res.samples.shape
    best_logp = float(np.nanmax(res.log_prob))
    print(f'[OK][mb ] sampler={sampler:7s} samples={ns} ndim={ndim} best_logp={best_logp:.3e}')
    return 'ok'


print('=== Multiband smoke test (model=nickel) ===')
counts_mb = {'ok': 0, 'skip': 0, 'fail': 0}
for name in ('emcee', 'zeus', 'dynesty'):
    st = run_one_multiband_sampler(name, sampler_cfg[name])
    counts_mb[st] += 1
print('Multiband summary:', counts_mb)


Initialising ensemble of 12 walkers...


=== Multiband smoke test (model=nickel) ===
[OK][mb ] sampler=emcee   samples=240 ndim=3 best_logp=-1.444e+07


[OK][mb ] sampler=zeus    samples=180 ndim=3 best_logp=-1.181e+07


/opt/anaconda3/lib/python3.12/site-packages/dynesty/dynesty.py:161: UserWarning: Specifying slice option while using rwalk sampler or  walks option with a slice sampler does not make sense
  warnings.warn('Specifying slice option while using rwalk sampler or '


[OK][mb ] sampler=dynesty samples=200 ndim=3 best_logp=-1.407e+07
Multiband summary: {'ok': 3, 'skip': 0, 'fail': 0}


/opt/anaconda3/lib/python3.12/site-packages/dynesty/sampler.py:1082: UserWarning: The sampling was stopped short due to maxiter/maxcall limit the delta(log(z)) criterion is not achieved; posterior may be poorly sampled
  warnings.warn('The sampling was stopped short due to'


In [5]:
print('=== Final summary ===')
print('bolometric:', counts_bol)
print('multiband :', counts_mb)


=== Final summary ===
bolometric: {'ok': 3, 'skip': 0, 'fail': 0}
multiband : {'ok': 3, 'skip': 0, 'fail': 0}
